# Forecast initial-shock decay timescales

Compares two forecast ensembles, caches regional shock anomalies and fitted decay timescales, and plots each selected request.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

season = "S0"
shock_experiment = "CAPT10-S0"
baseline_experiment = "CTRL10-S0"
variables = ("TS", "TREFHT", "PSL", "PRECT")
regions = ("global",)
max_members = None
force_compute = False
show = True


In [ ]:
import sys
import matplotlib.pyplot as plt
repo=WORK_DIR.resolve()
if str(repo/"diagnostic") not in sys.path: sys.path.insert(0,str(repo/"diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.forecast_regional_timeseries import REGIONS, VARIABLES, read_model_regional_series
from util.forecast_initial_shock import shock_cache_path, cache_shock_decay, plot_shock_decay


## Notes: available selections

Variables: `TS`, `TREFHT`, `PSL`, and `PRECT`. Regions: `global`, `CONUS`, `NHMidLat`, `Tropics`, `NINO3.4`, and `MAR_CONT`. The fitted diagnostic is `|shock ensemble mean - baseline ensemble mean| = a exp(-lead/tau) + c`; cached output contains signed and absolute anomalies, fitted magnitude, ensemble-mean `tau_days`, and member-level decay times.

The default compares CAPT initialization against CTRL. Existing caches are reused unless `force_compute = True`.


In [ ]:
workflow_name="fcst_atm_initial_shock"
data_dir,figure_dir=workflow_output_dirs(workflow_name,output_root=DIAGNOSTIC_OUTPUT_ROOT)
registry=experiment_registry(INPUT_DATA_ROOT,season,"fc"); period=registry[shock_experiment]["period"]
if registry[baseline_experiment]["period"]!=period: raise ValueError("Shock and baseline periods differ")
for variable in variables:
    for region in regions:
        target=shock_cache_path(data_dir,variable,region,shock_experiment,baseline_experiment,period); existed=target.is_file()
        if existed and not force_compute:
            ds=cache_shock_decay(target,force_compute=False); print("[REUSE]",target)
        else:
            shock,_=read_model_regional_series(INPUT_DATA_ROOT,shock_experiment,season,"fc",variable,region,period,max_members=max_members)
            baseline,_=read_model_regional_series(INPUT_DATA_ROOT,baseline_experiment,season,"fc",variable,region,period,max_members=max_members)
            ds=cache_shock_decay(target,shock=shock,baseline=baseline,force_compute=force_compute,shock_experiment=shock_experiment,baseline_experiment=baseline_experiment,variable=variable,region=region)
            print("[WRITE]",target)
        fig=plot_shock_decay(ds); output=figure_dir/f"{variable}_{region}_{shock_experiment}_minus_{baseline_experiment}_shock_decay.png"; fig.savefig(output,dpi=200,bbox_inches="tight")
        if not show: plt.close(fig)
        print("tau days:",float(ds.tau_days),"[FIGURE]",output)
